# ==============================================================================
# 🏭 ENVIRONMENTAL POLLUTION: CHEMICAL SENSOR AIR QUALITY VIA RANDOM FOREST
# ==============================================================================
### Core Principles:
Forecasting ground-truth Benzene (C6H6) concentration in $\mu\text{g}/\text{m}^3$ from an array of 5 metal oxide chemical sensor responses (CO, NMHC, NOx, NO2, O3, Temperature, RH, AH).
Random Forest captures non-linear chemical sensor responses and cross-sensitivities.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Chemical Sensor Environment ready.")


✅ STEP 1: Chemical Sensor Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Air Quality dataset (9,471 sensor records).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/air_quality/AirQuality.csv'
df = pd.read_csv(data_path, sep=';', decimal=',')

# Clean columns and drop empty metadata
df = df.dropna(how='all')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.drop(columns=['Date', 'Time'])

# -200 represents missing sensor readings in this dataset
df = df.replace(-200, np.nan)

# Drop columns with excessive missingness if any (e.g. NMHC)
df = df.dropna(subset=['C6H6(GT)']).copy()

df.columns = [c.strip().replace('(GT)', '_gt').replace('.', '_') for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} sensor records, {df.shape[1]} attributes")
print(f"Benzene Summary:\n{df['C6H6_gt'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 8,991 sensor records, 13 attributes
Benzene Summary:
count    8991.00
mean       10.08
std         7.45
min         0.10
25%         4.40
50%         8.20
75%        14.00
max        63.70
Name: C6H6_gt, dtype: float64


,CO_gt,PT08_S1(CO),NMHC_gt,C6H6_gt,PT08_S2(NMHC),NOx_gt,PT08_S3(NOx),NO2_gt,PT08_S4(NO2),PT08_S5(O3),T,RH,AH
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `C6H6_gt` (Benzene level).


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['C6H6_gt'])
y = df['C6H6_gt']

print(f"Missing values across sensors:\n{X.isnull().sum()[X.isnull().sum() > 0]}")


Missing values across sensors:
CO_gt      1647
NMHC_gt    8104
NOx_gt     1595
NO2_gt     1598
dtype: int64


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Readings: {X_train.shape[0]:,}")
print(f"Test Readings : {X_test.shape[0]:,}")


Train Readings: 7,192
Test Readings : 1,799


## ⚙️ STEP 7: PREPROCESSING PIPELINE
Median imputation for sensor dropouts.


In [5]:
# STEP 7: PREPROCESSOR
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
print("✅ STEP 7: Robust Sensor Imputation ready.")


✅ STEP 7: Robust Sensor Imputation ready.


## ⚠️ STEP 8: BASELINE FIT
Evaluating baseline Random Forest on sensor signals.


In [6]:
# STEP 8: BASELINE FIT
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)

r2_base = r2_score(y_test, pipe_rf.predict(X_test))
print(f"Baseline Random Forest R² : {r2_base*100:.2f}%")
print(f"Baseline RMSE             : {np.sqrt(mean_squared_error(y_test, pipe_rf.predict(X_test))):.2f} ug/m3")


Baseline Random Forest R² : 99.98%
Baseline RMSE             : 0.12 ug/m3


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `max_depth` and `min_samples_leaf`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [10, 14],
    'rf__min_samples_leaf': [2, 4]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])
grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²   : {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'rf__max_depth': 10, 'rf__min_samples_leaf': 2, 'rf__n_estimators': 150}
🎯 Best 5-Fold CV R²   : 99.93%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Tree)
Benchmarking ensemble gain against a single decision tree.


In [8]:
# STEP 10: BENCHMARK
single_tree = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(random_state=42))
])
single_tree.fit(X_train, y_train)

r2_tree = r2_score(y_test, single_tree.predict(X_test))
r2_rf = r2_score(y_test, best_model.predict(X_test))

print("="*65)
print(f"Single Decision Tree -> R²: {r2_tree*100:.2f}% | RMSE: {np.sqrt(mean_squared_error(y_test, single_tree.predict(X_test))):.2f} ug/m3")
print(f"Random Forest (Tuned)-> R²: {r2_rf*100:.2f}% | RMSE: {np.sqrt(mean_squared_error(y_test, best_model.predict(X_test))):.2f} ug/m3")
print("="*65)


Single Decision Tree -> R²: 99.98% | RMSE: 0.11 ug/m3
Random Forest (Tuned)-> R²: 99.99% | RMSE: 0.07 ug/m3


## 📊 STEP 11: RESIDUAL ANALYSIS & EVALUATION REPORT
Diagnostics on air quality errors.


In [9]:
# STEP 11: EVALUATION REPORT
y_pred_rf = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"📋 AIR TOXICITY DIAGNOSTIC METRICS:")
print(f"   Mean Absolute Error (MAE) : {mae:.2f} ug/m3")
print(f"   Root Mean Squared Error   : {rmse:.2f} ug/m3")
print(f"   R² Score                  : {r2_score(y_test, y_pred_rf)*100:.2f}%")


📋 AIR TOXICITY DIAGNOSTIC METRICS:
   Mean Absolute Error (MAE) : 0.01 ug/m3
   Root Mean Squared Error   : 0.07 ug/m3
   R² Score                  : 99.99%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & IOT SMOKE TEST
Deploying the sensor calibration pipeline and testing edge latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/air_quality_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_reading = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_benzene = loaded_pipe.predict(sample_reading)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏭 LIVE IOT SENSOR AIR QUALITY AUDIT:")
print(f"   Estimated Benzene Level: 🌫️ {pred_benzene:.2f} ug/m3")
print(f"   Actual Sensor Groundtruth: {y_test.iloc[0]:.2f} ug/m3")
print(f"   Sensor Inference Latency: {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/air_quality_random_forest_pipeline.joblib (3,059,537 bytes)

🏭 LIVE IOT SENSOR AIR QUALITY AUDIT:
   Estimated Benzene Level: 🌫️ 8.60 ug/m3
   Actual Sensor Groundtruth: 8.60 ug/m3
   Sensor Inference Latency: 50.188 ms (SLA < 2.0ms: CHECK)
